In [ ]:
## 7. De acá a la corrida completa: pasar a script con checkpointing
Si el prompt te convenció con la muestra, **no sigan corriendo la corrida
completa desde este notebook**. La idea de pasar a un script plano
(`generar_descripciones.py`) es que:

- Guarda cada descripción en la base relacional apenas se genera (no al
  final), así si se corta a la mitad no perdiste nada.
- Al reiniciarlo, salta automáticamente los `product_id` que ya tienen
  descripción guardada — no vuelve a gastar tokens de más.

Esqueleto de ese script (usando SQLite como base relacional simple, la que
sea más fácil como dice el doc):

```python
import sqlite3
import pandas as pd
from tqdm import tqdm
# ... importar generar_descripcion, SYSTEM_PROMPT, df igual que en este notebook

conn = sqlite3.connect("productos.db")
conn.execute("""
    CREATE TABLE IF NOT EXISTS descripciones (
        product_id TEXT PRIMARY KEY,
        descripcion TEXT,
        n_reviews_usadas INTEGER
    )
""")
conn.commit()

ya_procesados = {
    row[0] for row in conn.execute("SELECT product_id FROM descripciones")
}

todos_los_productos = df["ProductId"].unique()
pendientes = [pid for pid in todos_los_productos if pid not in ya_procesados]

print(f"Ya procesados: {len(ya_procesados)} | Pendientes: {len(pendientes)}")

for pid in tqdm(pendientes):
    descripcion, n_usadas = generar_descripcion(pid, df)
    if descripcion is not None:
        conn.execute(
            "INSERT OR REPLACE INTO descripciones VALUES (?, ?, ?)",
            (pid, descripcion, n_usadas),
        )
        conn.commit()  # commit por producto: más lento pero a prueba de cortes
```

Corriendo esto con `python generar_descripciones.py` en su compu local (no
hace falta notebook ni Colab para esta parte, es solo I/O de red), pueden
cortar el proceso en cualquier momento (`Ctrl+C`) y retomarlo después sin
duplicar trabajo ni gasto.
### Próximo paso
Con `productos.db` completo (tabla `descripciones`), el siguiente bloque es
el **paso 4 del documento**: embeddear cada `descripcion` y cargarla en
Chroma, para armar el retrieval de cara al usuario (capa 2) que ya diseñamos
en el notebook de chunking.
import os
import pandas as pd

CHECKPOINT_PATH = "descripciones_checkpoint.csv"
PARCIAL_PATH = "descripciones_parciales.csv"

if os.path.exists(CHECKPOINT_PATH):
    df_descripciones = pd.read_csv(CHECKPOINT_PATH)
elif os.path.exists(PARCIAL_PATH):
    df_descripciones = pd.read_csv(PARCIAL_PATH)
else:
    df_descripciones = pd.DataFrame(columns=["product_id", "descripcion", "n_reviews_usadas"])

if "descripcion" in df_descripciones.columns:
    df_descripciones["descripcion"] = df_descripciones["descripcion"].fillna("")
if "n_reviews_usadas" in df_descripciones.columns:
    df_descripciones["n_reviews_usadas"] = pd.to_numeric(
        df_descripciones["n_reviews_usadas"], errors="coerce"
    ).fillna(0).astype(int)

print(f"Descripciones cargadas: {len(df_descripciones)}")
print(f"Longitud promedio de descripción: {round(df_descripciones['descripcion'].str.len().mean(), 1)}")
print("\nDistribución de reviews usadas:")
print(df_descripciones["n_reviews_usadas"].value_counts().sort_index())

df_descripciones.head(10)

# Revisar ejemplos concretos para detectar descripciones demasiado genéricas o poco precisas
for pid in df_descripciones["product_id"].head(10).tolist():
    row = df_descripciones[df_descripciones["product_id"] == pid].iloc[0]
    print(f"\n=== {pid} ===")
    print(row["descripcion"])
    print("-" * 80)

# Prompt mejorado para generar descripciones más concretas y menos genéricas
SYSTEM_PROMPT_MEJORADO = """Sos un asistente que redacta descripciones breves y precisas de productos de supermercado a partir de reviews reales.

Reglas:
1. Usa solo la información presente en las reviews.
2. Destaca 2 o 3 atributos concretos: sabor, textura, uso, empaque, precio, ingredientes o problemas frecuentes.
3. Si hay opiniones contradictorias, menciona esa ambigüedad de forma breve.
4. Escribe en español, en 1 o 2 oraciones, con tono neutral y claro.
5. Evita frases vacías como "es una opción popular" o "muy valorado".
6. Si la información es limitada, agrega al final: "(Información limitada: pocas reviews disponibles)".
"""

SYSTEM_PROMPT_MEJORADO
# Crear un lookup de nombres inferidos por ProductId y unirlo al dataframe principal
if "product_name_inferred" not in df.columns:
    product_ids = df["ProductId"].dropna().unique()
    product_name_map = {
        pid: infer_product_name(pid, df, max_reviews=10)
        for pid in product_ids
    }
    df["product_name_inferred"] = df["ProductId"].map(product_name_map)

name_lookup = (
    df[["ProductId", "product_name_inferred"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

if "product_name_inferred" in df.columns:
    df = df.drop(columns=["product_name_inferred"])

# Merge al dataframe principal

df = df.merge(name_lookup, on="ProductId", how="left")

print(df[["ProductId", "product_name_inferred"]].drop_duplicates().head(10))
